# multivon-eval — Agent Evaluation

[![PyPI](https://img.shields.io/pypi/v/multivon-eval.svg)](https://pypi.org/project/multivon-eval)
[![Docs](https://img.shields.io/badge/docs-evaldocs.multivon.ai-violet)](https://evaldocs.multivon.ai)
[![GitHub](https://img.shields.io/badge/github-multivon--ai%2Fmultivon--eval-black)](https://github.com/multivon-ai/multivon-eval)

**multivon-eval** includes a full agent evaluation toolkit — trace recording, tool call accuracy, LLM judges for plan quality and task completion, and flakiness detection across runs.

| Part | What | API key? |
|------|------|---------|
| 1 | ManualTracer — record tool calls from any agent | ✓ |
| 2 | ToolCallAccuracy — expected tool sequences | ✗ |
| 3 | LLM judge — plan quality and task completion | ✓ |
| 4 | Multi-run flakiness detection | — |
| 5 | EvalSuite.for_agents() + add_check | ✓ |

Docs: [evaldocs.multivon.ai](https://evaldocs.multivon.ai) · GitHub: [multivon-ai/multivon-eval](https://github.com/multivon-ai/multivon-eval)

In [ ]:
!pip install 'multivon-eval>=0.7.0' anthropic -q

## API key setup

An Anthropic key is needed for Parts 1, 3, and 5 (agent calls + LLM judges). Parts 2 and 4 run without one.

In [ ]:
import os

os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # used for the agent and LLM judge

---
## Part 1 — ManualTracer: record tool calls from any agent

`ManualTracer` lets you instrument any agent — Anthropic, OpenAI, LangChain, custom — by wrapping tool calls in a context manager. The tracer captures the full execution trace (thoughts, tool inputs, tool outputs) so evaluators can analyze what the agent actually did.

**Scenario:** A customer support agent that handles order inquiries. It has access to three tools:
- `get_order_status` — look up order details
- `issue_refund` — trigger a refund
- `send_email` — send a confirmation to the customer

In [ ]:
import anthropic

# ---------------------------------------------------------------------------
# Simulated tool implementations (no real API calls)
# ---------------------------------------------------------------------------

def get_order_status(order_id: str) -> dict:
    """Return hardcoded order data for testing."""
    orders = {
        "ORD-001": {"status": "delivered", "item": "Laptop Stand", "total": 49.99},
        "ORD-002": {"status": "in_transit", "item": "Mechanical Keyboard", "total": 129.00},
        "ORD-003": {"status": "delayed",    "item": "USB-C Hub",           "total": 34.50},
    }
    return orders.get(order_id, {"status": "not_found"})

def issue_refund(order_id: str, amount: float) -> dict:
    """Simulate issuing a refund."""
    return {"refund_id": f"REF-{order_id}", "amount": amount, "status": "approved"}

def send_email(to: str, subject: str, body: str) -> dict:
    """Simulate sending an email."""
    return {"delivered": True, "to": to, "subject": subject}

# Tool dispatcher — called when the LLM requests a tool
TOOLS = {
    "get_order_status": lambda args: get_order_status(args["order_id"]),
    "issue_refund":     lambda args: issue_refund(args["order_id"], args["amount"]),
    "send_email":       lambda args: send_email(args["to"], args["subject"], args["body"]),
}

# Tool schemas passed to the Anthropic client
TOOL_SCHEMAS = [
    {
        "name": "get_order_status",
        "description": "Look up the status and details of a customer order.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string", "description": "The order ID"}},
            "required": ["order_id"],
        },
    },
    {
        "name": "issue_refund",
        "description": "Issue a refund for a given order.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "amount":   {"type": "number", "description": "Refund amount in USD"},
            },
            "required": ["order_id", "amount"],
        },
    },
    {
        "name": "send_email",
        "description": "Send an email to the customer.",
        "input_schema": {
            "type": "object",
            "properties": {
                "to":      {"type": "string"},
                "subject": {"type": "string"},
                "body":    {"type": "string"},
            },
            "required": ["to", "subject", "body"],
        },
    },
]

In [ ]:
import json
from multivon_eval import EvalSuite, EvalCase, ManualTracer

client = anthropic.Anthropic()
tracer = ManualTracer()

def support_agent(input_text: str) -> str:
    """
    Agentic loop: the model requests tools, we execute them and feed
    results back, repeating until the model produces a final text response.
    Each tool call is recorded with tracer.step() + s.record_tool_call().
    """
    messages = [{"role": "user", "content": input_text}]
    system = (
        "You are a customer support agent. Use tools to look up order information "
        "before taking any action. Always check order status before issuing a refund."
    )

    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            system=system,
            tools=TOOL_SCHEMAS,
            messages=messages,
        )

        if response.stop_reason == "end_turn":
            # Extract the final text reply
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text
            return ""

        # Process all tool_use blocks in this turn
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue

            tool_name   = block.name
            tool_input  = block.input
            tool_output = TOOLS[tool_name](tool_input)   # execute simulated tool

            # Record this tool call in the tracer
            with tracer.step(thought=f"Calling {tool_name}") as s:
                s.record_tool_call(tool_name, tool_input, tool_output)

            tool_results.append({
                "type":        "tool_result",
                "tool_use_id": block.id,
                "content":     json.dumps(tool_output),
            })

        # Feed tool results back for the next turn
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user",      "content": tool_results})


# Single case — refund request
suite = EvalSuite("Support Agent — basic trace")
suite.add_cases([
    EvalCase(
        input="My order ORD-003 is delayed. I want a refund. My email is alice@example.com.",
        expected_output="refund",   # rough expected content for sanity check
    ),
])

report = suite.run(support_agent, tracer=tracer)

In [ ]:
# Inspect the recorded trace for the first case
cr = report.case_results[0]
print(f"Input:  {cr.case_input}")
print(f"Output: {cr.actual_output[:200]}")
print(f"Passed: {cr.passed}")
print(f"Status: {cr.status}")
print()

if cr.agent_trace:
    print(f"Steps recorded: {len(cr.agent_trace)}")
    for i, step in enumerate(cr.agent_trace, 1):
        print(f"\n  Step {i}: {step.thought}")
        for tc in step.tool_calls:
            print(f"    tool:     {tc.name}")
            print(f"    args:     {tc.arguments}")
            print(f"    result:   {tc.result}")
        if step.output:
            print(f"    output:   {step.output[:100]}")


---
## Part 2 — ToolCallAccuracy: verify expected tool sequences

`ToolCallAccuracy` checks whether the agent called the expected tools. By default `require_order=False`, meaning the set of tools matters but the order doesn't. Set `require_order=True` for strict sequence checking — useful when tool order is semantically important (e.g., you must look up an order _before_ issuing a refund).

Attach expected tools to each case via `EvalCase(expected_tool_calls=[...])`.

In [ ]:
from multivon_eval import EvalSuite, EvalCase, ToolCallAccuracy

suite = EvalSuite("Support Agent — ToolCallAccuracy")
suite.add_evaluators(
    ToolCallAccuracy(require_order=False),  # set matters, order doesn't
)
suite.add_cases([
    # Refund request — expect status lookup then refund
    EvalCase(
        input="Order ORD-003 is delayed. Please refund me. Email: alice@example.com.",
        expected_tool_calls=["get_order_status", "issue_refund", "send_email"],
    ),
    # Status inquiry only — no refund expected
    EvalCase(
        input="Where is my order ORD-002?",
        expected_tool_calls=["get_order_status"],
    ),
    # Status + refund, no email required
    EvalCase(
        input="Order ORD-001 arrived damaged. Issue a refund please.",
        expected_tool_calls=["get_order_status", "issue_refund"],
    ),
    # Should only greet — no tools at all
    EvalCase(
        input="Hi, I'd just like to say your service is great!",
        expected_tool_calls=[],
    ),
])

report = suite.run(support_agent, tracer=tracer)

In [ ]:
# What tools were actually called vs what was expected?
print(f"Pass rate: {report.pass_rate:.0%}  ({report.evaluated}/{report.total} evaluated)\n")

for cr in report.case_results:
    status = "PASS" if cr.passed else "FAIL"
    print(f"[{status}] {cr.case_input[:60]}")

    if cr.agent_trace:
        actual_tools = [tc.name for step in cr.agent_trace for tc in step.tool_calls]
        print(f"       called:   {actual_tools}")

    for r in cr.results:
        if not r.passed:
            print(f"       ✗ {r.evaluator}: {r.reason[:80]}")


In [ ]:
# require_order=True — the agent MUST call tools in exactly the specified sequence.
# Useful for workflows where order is semantically critical.
suite_ordered = EvalSuite("Support Agent — strict order")
suite_ordered.add_evaluators(
    ToolCallAccuracy(require_order=True),
)
suite_ordered.add_cases([
    EvalCase(
        input="Order ORD-003 is delayed. Please refund me. Email: alice@example.com.",
        # Must look up status FIRST, then refund, then email — in this exact order
        expected_tool_calls=["get_order_status", "issue_refund", "send_email"],
    ),
])

report_ordered = suite_ordered.run(support_agent, tracer=tracer)
print(f"Ordered pass rate: {report_ordered.pass_rate:.0%}")
for cr in report_ordered.case_results:
    for r in cr.results:
        print(f"  {r.evaluator}: passed={r.passed}  reason={r.reason[:150]}")

---
## Part 3 — LLM judge: plan quality and task completion

`ToolCallAccuracy` tells you _which_ tools were called — but it can't tell you whether the agent's reasoning was sound or whether the final response actually helped the customer.

Two LLM-judge evaluators fill that gap:

- **`PlanQuality`** — did the agent plan its steps sensibly? Was the reasoning coherent before acting?
- **`TaskCompletion`** — did the agent actually complete the user's goal? Did the final response address what was asked?

Both use QAG scoring (binary yes/no questions) rather than unreliable 1–10 ratings.

In [ ]:
from multivon_eval import (
    EvalSuite, EvalCase,
    ToolCallAccuracy, PlanQuality, TaskCompletion,
)

suite = EvalSuite("Support Agent — LLM judges")
suite.add_evaluators(
    ToolCallAccuracy(require_order=False),
    PlanQuality(),     # LLM judge: was the agent's reasoning sound?
    TaskCompletion(),  # LLM judge: did the agent complete the task?
)
suite.add_cases([
    EvalCase(
        input="Order ORD-003 is delayed. I want a refund. My email is alice@example.com.",
        expected_tool_calls=["get_order_status", "issue_refund", "send_email"],
    ),
    EvalCase(
        input="Where is my order ORD-002? I need it for a meeting tomorrow.",
        expected_tool_calls=["get_order_status"],
    ),
])

report = suite.run(support_agent, tracer=tracer)

In [ ]:
# See scores broken out by evaluator
print("Scores by evaluator:")
for name, score in report.scores_by_evaluator().items():
    print(f"  {name}: {score:.2f}")

print()
for cr in report.case_results:
    status = "PASS" if cr.passed else "FAIL"
    print(f"[{status}] {cr.case_input[:70]}")
    for r in cr.results:
        icon = "✓" if r.passed else "✗"
        print(f"  {icon} {r.evaluator:<22} score={r.score:.2f}")
        if r.reason:
            print(f"      {r.reason[:200]}")
    print()

### When tool calls are right but the response is wrong

A critical use case for LLM judges: an agent can call all the right tools and still produce a poor final response — for example, giving a vague non-answer after successfully issuing a refund. `ToolCallAccuracy` passes; `TaskCompletion` catches it.

The cell below injects a deliberately bad agent response to demonstrate this divergence.

In [ ]:
# Wrap the real agent with a bad final-response stub
# The agent correctly calls get_order_status + issue_refund, but returns
# a useless reply — TaskCompletion should fail even though tools are correct.

def support_agent_bad_response(input_text: str) -> str:
    """Calls correct tools but returns a deliberately unhelpful response."""
    # Run the real agentic loop to generate the tool trace
    support_agent(input_text)   # side-effect: tracer is populated
    # Override the final reply with something useless
    return "We have processed your request. Please check back later."


suite_div = EvalSuite("Divergence demo — tools right, response wrong")
suite_div.add_evaluators(
    ToolCallAccuracy(require_order=False),
    TaskCompletion(),
)
suite_div.add_cases([
    EvalCase(
        input="Order ORD-003 is delayed. Please refund me.",
        expected_tool_calls=["get_order_status", "issue_refund"],
    ),
])

report_div = suite_div.run(support_agent_bad_response, tracer=tracer)

print("Evaluator breakdown:")
for cr in report_div.case_results:
    for r in cr.results:
        icon = "✓" if r.passed else "✗"
        print(f"  {icon} {r.evaluator:<22} passed={r.passed}")
        if r.reason:
            print(f"      {r.reason[:250]}")

# Expected: ToolCallAccuracy=PASS, TaskCompletion=FAIL

---
## Part 4 — Multi-run flakiness detection

LLMs are non-deterministic. An agent may call `get_order_status` on one run and skip it on another. A single-run eval score can be noise.

`runs=N` re-executes each case N times and aggregates results. Cases with inconsistent pass/fail outcomes across runs are flagged as **flaky**.

Key metrics:
- **`report.stability_score`** — fraction of cases that are stable (same result every run)
- **`cr.run_pass_rate`** — how often this specific case passed across all runs
- **`cr.is_flaky`** — True if the case neither always passed nor always failed

`early_stop=True` uses SPRT (Sequential Probability Ratio Test) to stop early once the result for a case is statistically clear — saving LLM calls on easy cases.

In [ ]:
from multivon_eval import EvalSuite, EvalCase, ToolCallAccuracy

suite = EvalSuite("Support Agent — flakiness")
suite.add_evaluators(ToolCallAccuracy(require_order=False))
suite.add_cases([
    EvalCase(
        input="Order ORD-003 is delayed. Please refund me. Email: alice@example.com.",
        expected_tool_calls=["get_order_status", "issue_refund", "send_email"],
    ),
    EvalCase(
        input="Where is my order ORD-002?",
        expected_tool_calls=["get_order_status"],
    ),
    EvalCase(
        input="Hi, your service is great — no questions today!",
        expected_tool_calls=[],
    ),
])

# Run each case 5 times; use SPRT to cut off early on clear-cut cases
report = suite.run(support_agent, tracer=tracer, runs=5, early_stop=True)

print(f"Flaky cases:     {report.flaky_count} / {report.total}")
print(f"Stability score: {report.stability_score:.0%}")
print(f"Avg runs used:   ~{report.runs_per_case:.1f} (budget: 5)")

In [ ]:
# Drill into per-case pass rates to find which tool calls are inconsistent
print("Per-case breakdown:")
for cr in report.case_results:
    stability = "STABLE" if not cr.is_flaky else "FLAKY "
    print(f"  [{stability}] pass_rate={cr.run_pass_rate:.0%}  input={cr.case_input[:55]}")

print()
print("Flaky cases to investigate:")
flaky = [cr for cr in report.case_results if cr.is_flaky]
if flaky:
    for cr in flaky:
        print(f"  Input: {cr.case_input}")
        print(f"  The agent inconsistently picks different tools on different runs.")
        print(f"  Consider adding clearer instructions or pinning tool-choice logic.")
else:
    print("  None — all cases are stable across runs.")

---
## Part 5 — EvalSuite.for_agents() + add_check

`EvalSuite.for_agents()` is a factory that pre-loads all agent evaluators in one call:
- `ToolCallAccuracy(require_order=False)`
- `ToolCallNecessity` — flags unnecessary tool calls (agent called a tool it didn't need)
- `TrajectoryEfficiency` — penalises redundant or repeated steps
- `PlanQuality` — LLM judge for reasoning quality
- `TaskCompletion` — LLM judge for goal completion

You can still call `add_check()` and `add_evaluators()` on top to add extra criteria.

In [ ]:
from multivon_eval import EvalSuite, EvalCase

# One-line setup — all agent evaluators included
suite = EvalSuite.for_agents(require_order=False)

suite.add_cases([
    EvalCase(
        input="Order ORD-003 is delayed. Refund me please. Email: alice@example.com.",
        expected_tool_calls=["get_order_status", "issue_refund", "send_email"],
    ),
    EvalCase(
        input="Where is order ORD-002?",
        expected_tool_calls=["get_order_status"],
    ),
])

report = suite.run(support_agent, tracer=tracer)

print(f"Pass rate: {report.pass_rate:.0%}")
print("Evaluators included:")
for name in report.scores_by_evaluator():
    print(f"  {name}")

### Behavioral guardrails with add_check

`ToolCallAccuracy` tells you _which_ tools were called. `add_check` lets you describe _how_ the agent should behave in plain English — guardrails that tool-call lists can't express.

In [ ]:
from multivon_eval import EvalSuite, EvalCase

suite = EvalSuite.for_agents(require_order=False)

# Plain-English behavioral guardrails — things tool-call lists can't verify
suite.add_check(
    "Agent should acknowledge the customer's problem before taking action"
)
suite.add_check(
    "Agent should not issue a refund without first checking order status"
)
suite.add_check(
    "Agent should confirm the refund amount and reference number in the reply"
)

suite.add_cases([
    EvalCase(
        input="Order ORD-003 is delayed. Please refund me. Email: alice@example.com.",
        expected_tool_calls=["get_order_status", "issue_refund", "send_email"],
    ),
    EvalCase(
        input="My order ORD-001 arrived damaged. I want my money back.",
        expected_tool_calls=["get_order_status", "issue_refund"],
    ),
])

report = suite.run(support_agent, tracer=tracer)

print(f"Pass rate: {report.pass_rate:.0%}\n")
for cr in report.case_results:
    status = "PASS" if cr.passed else "FAIL"
    print(f"[{status}] {cr.case_input[:65]}")
    for r in cr.results:
        icon = "✓" if r.passed else "✗"
        print(f"  {icon} {r.evaluator:<30} score={r.score:.2f}")
        if not r.passed and r.reason:
            print(f"      {r.reason[:200]}")
    print()

---
## Next steps

- **Docs:** [evaldocs.multivon.ai](https://evaldocs.multivon.ai)
- **Agent evaluator reference:** ToolCallAccuracy, ToolCallNecessity, TrajectoryEfficiency, PlanQuality, TaskCompletion → [Agent Evaluators](https://evaldocs.multivon.ai/evaluators/agent)
- **ManualTracer guide:** full API reference and multi-tool-per-step examples → [ManualTracer docs](https://evaldocs.multivon.ai/tracers/manual)
- **Compliance:** HIPAA PII detection, EU AI Act audit trails (especially relevant for agents that handle customer PII) → [Compliance Guide](https://evaldocs.multivon.ai/guides/compliance)
- **CI/CD integration:** block deploys when agent accuracy regresses → [CI/CD Guide](https://evaldocs.multivon.ai/guides/ci-cd)
- **Experiment tracking:** compare agent versions with p-values → [Experiment Tracking](https://evaldocs.multivon.ai/guides/experiments)
- **GitHub:** [github.com/multivon-ai/multivon-eval](https://github.com/multivon-ai/multivon-eval)

**Found a bug or want a feature?** Open an issue: [github.com/multivon-ai/multivon-eval/issues](https://github.com/multivon-ai/multivon-eval/issues)